<a href="https://colab.research.google.com/github/GH-Rumi/PemMes_7_Helmi/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Install Library**

In [4]:
!pip install annoy faiss-cpu hnswlib numpy pandas

# **2. Import Library**

In [5]:
import time
import numpy as np
import pandas as pd
from annoy import AnnoyIndex
import faiss
import hnswlib

# **3. Kode Python Lengkap**

In [6]:
# Konfigurasi percobaan
SIZES = [1_000, 1_000_000]
DIMS = [2, 5]
METRICS = ["euclidean", "cosine"]
K = 10
N_QUERIES = 100

def get_ground_truth(data, queries, k, metric):
    """Menghitung tetangga terdekat eksak (Exact Search) sebagai acuan Recall."""
    dim = data.shape[1]
    if metric == "euclidean":
        index = faiss.IndexFlatL2(dim)
        index.add(data)
        _, gt_indices = index.search(queries, k)
    else:  # cosine (menggunakan Inner Product pada vektor ternormalisasi)
        data_norm = data.copy()
        queries_norm = queries.copy()
        faiss.normalize_L2(data_norm)
        faiss.normalize_L2(queries_norm)
        index = faiss.IndexFlatIP(dim)
        index.add(data_norm)
        _, gt_indices = index.search(queries_norm, k)
    return gt_indices

def calculate_recall(gt_indices, pred_indices, k):
    """Menghitung persentase Recall@K."""
    recalls = []
    for gt, pred in zip(gt_indices, pred_indices):
        matches = len(set(gt).intersection(set(pred)))
        recalls.append(matches / k)
    return np.mean(recalls) * 100

# --- 1. FUNGSI BENCHMARK ANNOY ---
def run_annoy(data, queries, k, metric, gt_indices):
    dim = data.shape[1]
    annoy_metric = "euclidean" if metric == "euclidean" else "angular"

    # Build Index
    t0 = time.perf_counter()
    index = AnnoyIndex(dim, annoy_metric)
    for i, vec in enumerate(data):
        index.add_item(i, vec)
    index.build(10)  # 10 trees
    build_time = time.perf_counter() - t0

    # Query
    t0 = time.perf_counter()
    pred_indices = [index.get_nns_by_vector(q, k) for q in queries]
    query_time = (time.perf_counter() - t0) * 1000  # dalam milidetik (ms)

    recall = calculate_recall(gt_indices, pred_indices, k)
    return build_time, query_time, recall

# --- 2. FUNGSI BENCHMARK FAISS (IVFFlat) ---
def run_faiss(data, queries, k, metric, gt_indices):
    n_data, dim = data.shape
    nlist = 10 if n_data <= 10_000 else 100  # jumlah cluster Voronoi

    if metric == "euclidean":
        d_use, q_use = data, queries
        quantizer = faiss.IndexFlatL2(dim)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
    else:
        d_use, q_use = data.copy(), queries.copy()
        faiss.normalize_L2(d_use)
        faiss.normalize_L2(q_use)
        quantizer = faiss.IndexFlatIP(dim)
        index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)

    # Build Index (Train + Add)
    t0 = time.perf_counter()
    index.train(d_use)
    index.add(d_use)
    build_time = time.perf_counter() - t0

    # Query
    index.nprobe = 5 if n_data <= 10_000 else 10
    t0 = time.perf_counter()
    _, pred_indices = index.search(q_use, k)
    query_time = (time.perf_counter() - t0) * 1000  # dalam milidetik (ms)

    recall = calculate_recall(gt_indices, pred_indices, k)
    return build_time, query_time, recall

# --- 3. FUNGSI BENCHMARK HNSW (hnswlib) ---
def run_hnsw(data, queries, k, metric, gt_indices):
    n_data, dim = data.shape
    hnsw_metric = "l2" if metric == "euclidean" else "cosine"

    # Build Index
    t0 = time.perf_counter()
    index = hnswlib.Index(space=hnsw_metric, dim=dim)
    index.init_index(max_elements=n_data, ef_construction=100, M=16)
    index.add_items(data)
    build_time = time.perf_counter() - t0

    # Query
    index.set_ef(50)
    t0 = time.perf_counter()
    pred_indices, _ = index.knn_query(queries, k=k)
    query_time = (time.perf_counter() - t0) * 1000  # dalam milidetik (ms)

    recall = calculate_recall(gt_indices, pred_indices, k)
    return build_time, query_time, recall

# --- EKSEKUSI UTAMA ---
results = []
np.random.seed(42)

for metric in METRICS:
    print(f"\nMenjalankan pengujian untuk metric: {metric.upper()}...")
    for n_data in SIZES:
        for dim in DIMS:
            # Generate dataset float32 (wajib float32 untuk FAISS & HNSW)
            data = np.random.random((n_data, dim)).astype("float32")
            queries = np.random.random((N_QUERIES, dim)).astype("float32")

            gt = get_ground_truth(data, queries, K, metric)

            b_annoy, q_annoy, r_annoy = run_annoy(data, queries, K, metric, gt)
            b_faiss, q_faiss, r_faiss = run_faiss(data, queries, K, metric, gt)
            b_hnsw, q_hnsw, r_hnsw = run_hnsw(data, queries, K, metric, gt)

            results.append({
                "Metric": metric.capitalize(),
                "Jumlah Data/Dimensi": f"{n_data}/{dim}D",
                "ANNOY (Build | Query | Recall)": f"{b_annoy:.3f}s | {q_annoy:.2f}ms | {r_annoy:.1f}%",
                "FAISS (Build | Query | Recall)": f"{b_faiss:.3f}s | {q_faiss:.2f}ms | {r_faiss:.1f}%",
                "HNSW (Build | Query | Recall)": f"{b_hnsw:.3f}s | {q_hnsw:.2f}ms | {r_hnsw:.1f}%"
            })

# Tampilkan hasil per metric dalam bentuk tabel rapi
df_results = pd.DataFrame(results)
for metric in METRICS:
    print(f"\n=== HASIL PERCOBAAN METRIC: {metric.upper()} ===")
    subset = df_results[df_results["Metric"] == metric.capitalize()].drop(columns=["Metric"])
    print(subset.to_string(index=False))


Menjalankan pengujian untuk metric: EUCLIDEAN...

Menjalankan pengujian untuk metric: COSINE...

=== HASIL PERCOBAAN METRIC: EUCLIDEAN ===
Jumlah Data/Dimensi ANNOY (Build | Query | Recall) FAISS (Build | Query | Recall) HNSW (Build | Query | Recall)
            1000/2D        0.044s | 3.21ms | 99.4%       0.011s | 0.58ms | 100.0%      0.027s | 1.51ms | 100.0%
            1000/5D        0.017s | 2.12ms | 96.4%        0.001s | 0.63ms | 99.8%      0.033s | 1.79ms | 100.0%
         1000000/2D       26.420s | 5.44ms | 99.8%      0.254s | 71.21ms | 100.0%     52.292s | 2.60ms | 100.0%
         1000000/5D       21.757s | 5.67ms | 96.3%      0.182s | 69.45ms | 100.0%     84.390s | 3.49ms | 100.0%

=== HASIL PERCOBAAN METRIC: COSINE ===
Jumlah Data/Dimensi ANNOY (Build | Query | Recall) FAISS (Build | Query | Recall) HNSW (Build | Query | Recall)
            1000/2D        0.059s | 2.09ms | 99.8%       0.001s | 0.69ms | 100.0%      0.023s | 0.93ms | 100.0%
            1000/5D        0.022s | 

### Hasil Percobaan Metric: EUCLIDEAN

| Jumlah Data/Dimensi | ANNOY (Build \| Query \| Recall) | FAISS (Build \| Query \| Recall) | HNSW (Build \| Query \| Recall) |
| :--- | :--- | :--- | :--- |
| **1000/2D** | 0.044s \| 3.21ms \| 99.4% | 0.011s \| 0.58ms \| 100.0% | 0.027s \| 1.51ms \| 100.0% |
| **1000/5D** | 0.017s \| 2.12ms \| 96.4% | 0.001s \| 0.63ms \| 99.8% | 0.033s \| 1.79ms \| 100.0% |
| **1000000/2D** | 26.420s \| 5.44ms \| 99.8% | 0.254s \| 71.21ms \| 100.0% | 52.292s \| 2.60ms \| 100.0% |
| **1000000/5D** | 21.757s \| 5.67ms \| 96.3% | 0.182s \| 69.45ms \| 100.0% | 84.390s \| 3.49ms \| 100.0% |

### Hasil Percobaan Metric: COSINE

| Jumlah Data/Dimensi | ANNOY (Build \| Query \| Recall) | FAISS (Build \| Query \| Recall) | HNSW (Build \| Query \| Recall) |
| :--- | :--- | :--- | :--- |
| **1000/2D** | 0.059s \| 2.09ms \| 99.8% | 0.001s \| 0.69ms \| 100.0% | 0.023s \| 0.93ms \| 100.0% |
| **1000/5D** | 0.022s \| 2.21ms \| 97.9% | 0.001s \| 0.74ms \| 99.9% | 0.032s \| 1.42ms \| 100.0% |
| **1000000/2D** | 88.424s \| 6.65ms \| 5.8% | 1.352s \| 168.94ms \| 99.4% | 85.686s \| 3.44ms \| 31.5% |
| **1000000/5D** | 27.677s \| 5.85ms \| 97.5% | 0.288s \| 98.16ms \| 100.0% | 79.865s \| 3.77ms \| 100.0% |